# Step 26 — An exactly solvable test bed for the tilt and the committor estimators (CPU, NumPy)

A six-class Gaussian mixture in $d=32$, with two superclasses of three classes each and class-specific widths. Its score, posterior and committor are all known in closed form. The "data" has uniform class weights; the "model" has the same class shapes with imbalanced weights, so it has a pure mass error.

**A. Band-limited tilts (reviewer concerns 6 and 7).** We sample the model with the exact reverse SDE and add a class tilt towards uniform. The tilt is computed three ways:
1. exactly, with the model's own posterior;
2. with the data posterior and $r=f^t/f^m$, which is the first-order form used on CIFAR-10;
3. with the data posterior shifted by $\log(f^m/f^d)$, which is exact.

Each is applied along the whole trajectory, or only above or only below a split point $\sigma_1\in\{0.5,1,2,4\}$, with three seeds and 20,000 samples. Do the high-band and low-band shares add up? Does the first-order tilt leave a residual?

**B. Committor estimators (concern 4).** Branching on the model with 400 trajectories, 12 levels and 12 continuations, against the *exact* committor (the model's posterior). For every pair, decisions from the eligibility estimator and the pooled cross-over estimator are compared with their infinite-sample targets and with a per-state definition computed from exact committors. We also check directly whether pairwise decisions can precede the superclass decision.

In [ ]:
import numpy as np, time
from itertools import combinations
rng = np.random.default_rng(26); d, Cn = 32, 6
SUPER = np.array([0, 0, 0, 1, 1, 1])
MU = np.zeros((Cn, d)); tri = np.array([[1, 0], [-0.5, np.sqrt(3)/2], [-0.5, -np.sqrt(3)/2]])
for c in range(Cn):
    MU[c, 0] = 3.0*(1 if SUPER[c] == 0 else -1); MU[c, 1:3] = 1.6*tri[c % 3]; MU[c, 3 + c] = 0.6          # superclass axis, within-superclass triangle, a small private direction
S2 = np.array([0.30, 0.45, 0.60, 0.35, 0.50, 0.65])**2                                                     # class variances
F_D = np.ones(Cn)/Cn; F_M = np.array([0.22, 0.12, 0.16, 0.20, 0.10, 0.20])
MU2 = (MU**2).sum(1)
def logits(x, s, f):
    v = S2 + s**2; d2 = (x*x).sum(1, keepdims=True) - 2*x@MU.T + MU2[None]
    return np.log(f)[None] - 0.5*d2/v[None] - 0.5*d*np.log(v)[None]
def post(x, s, f): l = logits(x, s, f); l -= l.max(1, keepdims=True); p = np.exp(l); return p/p.sum(1, keepdims=True)
def _mix(W, x, s):
    """sum_c W_c (mu_c - x)/v_c for weights W [N, C]"""
    v = S2 + s**2; Wv = W/v[None]; return Wv@MU - x*Wv.sum(1, keepdims=True)
def score(x, s, f): return _mix(post(x, s, f), x, s)
def tilt_grad(x, s, P, r):
    """grad log sum_c r_c P_c(x) for a posterior P computed with ANY class prior (the shapes are shared)"""
    q = r[None]*P; q = q/q.sum(1, keepdims=True); return _mix(q - P, x, s)
def sched(n=256, smax=40.0, smin=0.02, rho=7.0): i = np.arange(n); return (smax**(1/rho) + i/(n-1)*(smin**(1/rho) - smax**(1/rho)))**rho
SCH = sched()
def run(N, seed, mode=None, band=None, s1=None, x0=None, i0=0, keep=None):
    """Reverse SDE of the MODEL mixture. mode: None | 'exact' | 'first_order' | 'shifted'; band: None | 'high' | 'low' (split at s1)."""
    g = np.random.default_rng(seed); x = g.standard_normal((N, d))*SCH[0] if x0 is None else x0.copy(); states = {}
    r = F_D/F_M
    for i in range(i0, len(SCH)):
        s0 = SCH[i]; s_1 = SCH[i+1] if i+1 < len(SCH) else 0.0
        if keep is not None and i in keep: states[i] = x.copy()
        if s_1 <= 0: break
        sc = score(x, s0, F_M)
        on = mode is not None and (band is None or (band == 'high' and s0 >= s1) or (band == 'low' and s0 < s1))
        if on:
            if mode == 'exact': sc = sc + tilt_grad(x, s0, post(x, s0, F_M), r)
            elif mode == 'first_order': sc = sc + tilt_grad(x, s0, post(x, s0, F_D), r)
            elif mode == 'shifted':      # data posterior shifted by log(f_m/f_d) == model posterior; weight f_t/f_m
                Pd = post(x, s0, F_D)*(F_M/F_D)[None]; Pd = Pd/Pd.sum(1, keepdims=True); sc = sc + tilt_grad(x, s0, Pd, r)
        x = x + (s_1 - s0)*(-2*s0*sc) + np.sqrt(2*s0*(s0 - s_1))*g.standard_normal(x.shape)
    lab = post(x, SCH[-1], F_M).argmax(1)
    return (lab, states) if keep is not None else lab
ferr = lambda lab: np.mean(np.abs(np.log(np.maximum(np.bincount(lab, minlength=Cn)/len(lab), 1e-4)/F_D)))
print('model frequency error (no tilt):', round(ferr(run(20000, 0)), 4), '| target 0')

## A. Band-limited tilts

In [ ]:
N = 20000; SEEDS = [1, 2, 3]; t0 = time.time()
E0 = np.mean([ferr(run(N, s)) for s in SEEDS])
full = {m: np.mean([ferr(run(N, s, m)) for s in SEEDS]) for m in ['exact', 'first_order', 'shifted']}
print(f"untilted error {E0:.4f};  full-trajectory tilt: exact {full['exact']:.4f}, first-order (data posterior) {full['first_order']:.4f}, shifted {full['shifted']:.4f}")
print(f"\n{'split sigma_1':>13} {'high-only share':>18} {'low-only share':>16} {'sum':>8}   (share of the exact full-trajectory correction, mean +- sd over seeds)")
for s1 in [0.5, 1.0, 2.0, 4.0]:
    sh = {}
    for band in ['high', 'low']:
        e = np.array([ferr(run(N, s, 'exact', band, s1)) for s in SEEDS]); sh[band] = (E0 - e)/(E0 - full['exact'])
    print(f"{s1:13.1f} {sh['high'].mean():+10.2f} +- {sh['high'].std():.2f} {sh['low'].mean():+8.2f} +- {sh['low'].std():.2f} {sh['high'].mean()+sh['low'].mean():+8.2f}")
P_ = post(run(4000, 9, keep={int(np.argmin(np.abs(SCH - s))) for s in [0.5, 1, 2, 4]})[1][int(np.argmin(np.abs(SCH - 1.0)))], 1.0, F_M)
print(f"\n(for reference: mean max-posterior at sigma=1 along model trajectories = {P_.max(1).mean():.2f})  [{(time.time()-t0)/60:.1f} min]")

## B. Committor estimators against the exact committor

In [ ]:
M, K = 400, 12; LEVS = sorted({int(np.argmin(np.abs(SCH - s))) for s in np.geomspace(8.0, 0.1, 12)}); LSIG = SCH[LEVS]
PAR, ST = run(M, 101, keep=set(LEVS))
END = np.zeros((len(LEVS), M, K), int); QX = np.zeros((len(LEVS), M, Cn))
for li, i0 in enumerate(LEVS):
    x0 = np.repeat(ST[i0], K, axis=0); END[li] = run(M*K, 200 + li, x0=x0, i0=i0).reshape(M, K); QX[li] = post(ST[i0], SCH[i0], F_M)     # exact committor = model posterior
PAIRS = list(combinations(range(Cn), 2)); kind = lambda p: 'super' if SUPER[p[0]] != SUPER[p[1]] else 'within'
def first_level(vals, thr, above=True):
    ok = np.isfinite(vals); good = ok & ((vals >= thr) if above else (vals <= thr))
    for l in range(len(vals)):
        if good[l] and all(good[k] or not ok[k] for k in range(l, len(vals))): return LSIG[l]
    return np.nan
def eligibility(a, b, J, min_n=6):
    Cc = []
    for l in range(len(LEVS)):
        com = [((END[l, j] == PAR[j]).sum()/((END[l, j] == a).sum() + (END[l, j] == b).sum())) >= 0.9 for j in J if (END[l, j] == a).sum() + (END[l, j] == b).sum() >= min_n]
        Cc.append(np.mean(com) if len(com) >= 8 else np.nan)
    return first_level(np.array(Cc), 0.5)
def pooled(a, b, J):
    X = []
    for l in range(len(LEVS)):
        same = sum((END[l, j] == PAR[j]).sum() for j in J); cross = sum((END[l, j] == (b if PAR[j] == a else a)).sum() for j in J)
        X.append(cross/(same + cross) if same + cross >= 20 else np.nan)
    return first_level(np.array(X), 0.1, above=False)
def pooled_exact(a, b, J):
    X = []
    for l in range(len(LEVS)):
        q = QX[l, J]; own = np.where(PAR[J] == a, q[:, a], q[:, b]); oth = np.where(PAR[J] == a, q[:, b], q[:, a]); X.append(oth.sum()/(own + oth).sum())
    return first_level(np.array(X), 0.1, above=False)
def per_state_exact(a, b, J):
    Cc = []
    for l in range(len(LEVS)):
        q = QX[l, J]; own = np.where(PAR[J] == a, q[:, a], q[:, b]); Cc.append(np.mean(own/(q[:, a] + q[:, b]) >= 0.9))
    return first_level(np.array(Cc), 0.5)
print(f"{'pair':>6} {'kind':>7} {'eligibility (K=12)':>19} {'pooled (K=12)':>14} {'pooled, exact':>14} {'per-state, exact':>17}")
R = []
for a, b in PAIRS:
    J = np.where(np.isin(PAR, [a, b]))[0]
    r = (eligibility(a, b, J), pooled(a, b, J), pooled_exact(a, b, J), per_state_exact(a, b, J)); R.append(r)
    print(f"{a}/{b}:{'':>2} {kind((a, b)):>7} {r[0]:19.2f} {r[1]:14.2f} {r[2]:14.2f} {r[3]:17.2f}")
R = np.array(R); ok = np.all(np.isfinite(R), 1)
for j, nm in [(0, 'eligibility'), (1, 'pooled')]:
    tgt = 3 if j == 0 else 2
    print(f"{nm:>12}: median log-ratio to its exact target {np.median(np.log(R[ok, j]/R[ok, tgt])):+.3f}; to the per-state exact decision {np.median(np.log(R[ok, j]/R[ok, 3])):+.3f}")
supd = first_level(np.array([np.mean([(SUPER[END[l, j]] == SUPER[PAR[j]]).mean() >= 11/12 for j in range(M)]) for l in range(len(LEVS))]), 0.5)
print(f"population superclass decision (half of trajectories decided): {supd:.2f}; within-superclass pairs decided earlier than this (exact per-state): "
      f"{sum(1 for (p, r) in zip(PAIRS, R) if kind(p) == 'within' and r[3] > supd)}/{sum(1 for p in PAIRS if kind(p) == 'within')}")

## What to report back
- **Part A:** the untilted/full line and the split table.
- **Part B:** the pair table, the two bias lines, and the superclass line.